[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [SQLAlchemy, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlalchemy-deep-dive.html)

# Migrations with Alembic


## What you will be able to do

Set up Alembic for a database that already has tables, write revisions from the models with
`alembic revision --autogenerate`, read them before they run, and apply and undo them with `upgrade`
and `downgrade`. Change a SQLite table in batch mode, write the SQL of a migration without a
database, and recognize a database that is behind its revisions, a constraint SQLite cannot add in
place, a rename that autogenerate reads as a drop and an add, and two revisions written from the
same parent.


## The idea

### The problem

The college's database is in use, and the models keep changing: students get a phone number, the
college hires advisors, a report needs an index. `create_all` creates tables that are missing and
never touches a table that exists, so none of these reach the database through it. An `ALTER TABLE`
typed by hand reaches one database, on one day, and leaves no record: the copy on another laptop, the
test database and the production one each have to be changed the same way, in the same order, by
someone who remembers how.

A migration tool keeps those changes as files. Every change is a revision with an `upgrade()` that
makes it and a `downgrade()` that takes it back, every revision names the one before it, and every
database records which revision it has reached. Writing the files by hand is slow and easy to get
wrong, and a tool that writes them from the models can be wrong too: it can mistake a renamed
column for a new one, and write a migration that drops the old column with its data.

### What Alembic is

> **Alembic** is the migration tool from SQLAlchemy's authors. **`alembic init`** makes a
> **migration environment**: `alembic.ini`, its settings, and a folder with `env.py`, which runs
> every command, and `versions`, where the revisions go. A **revision** is a Python file with an id,
> the id of the revision before it, and `upgrade()` and `downgrade()` functions written with `op`,
> Alembic's operations. **`alembic revision --autogenerate`** writes one by comparing the models'
> `MetaData` with the database. **`alembic upgrade head`** runs every revision the database has not
> had, **`downgrade`** runs them backwards, and the table `alembic_version` holds the revision the
> database is at. **Batch mode**, `op.batch_alter_table`, changes a SQLite table by copying it.

### Why it works that way

- **A migration is code, reviewed like code.** The revision files go into version control beside the
  models, so every copy of the database gets the same changes, in the same order.
- **The database knows where it is.** `alembic_version` makes `upgrade head` run only what is
  missing, however many revisions behind a copy of the database is.
- **Autogenerate compares, and a person decides.** Alembic sees what differs between the models and
  the database, not why, so a rename looks like one column gone and another new. The file is a
  draft, to read before it runs.
- **SQLite's `ALTER TABLE` does little.** It can add a column, rename one and drop one, and no more,
  as the **Changing a Schema** notebook of the **sqlite3, Deep Dive** guide showed, so any other
  change to a table means building a new one and copying the rows across, which batch mode does.
- **Revisions form a chain.** Two revisions written from the same parent give the chain two heads,
  and Alembic will not guess which comes first.

### Where this shows up

Every project whose database outlives one version of its code needs migrations, and Alembic is the
tool for projects that use SQLAlchemy. The **Changing a Schema** notebook of the
**sqlite3, Deep Dive** guide changed tables by hand, the copy-and-rename that batch mode automates.
The **Four Databases, One Codebase** notebook writes this notebook's migrations as SQL for four
other databases, and **A Complete Data Layer** starts its database from a migration.

### What this notebook covers

- A migration environment for the college
- A first revision: the database as it is
- A new column, from the model to the database
- Down and up again
- Batch mode: a foreign key SQLite cannot add in place
- The SQL without the database: `--sql`
- Which command for which job
- A schema change in one call, finished
- Four errors, from a database behind its revisions to two heads at once

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import sqlalchemy as sa
from alembic.migration import MigrationContext
from alembic.operations import Operations

engine = sa.create_engine("sqlite://")
with engine.begin() as conn:
    conn.execute(sa.text("CREATE TABLE students (id INTEGER PRIMARY KEY, name TEXT)"))
    op = Operations(MigrationContext.configure(conn))
    op.add_column("students", sa.Column("phone", sa.String(20)))
    columns = sa.inspect(conn).get_columns("students")
print([column["name"] for column in columns])
```

```
['id', 'name', 'phone']
```

`op.add_column` is one of Alembic's operations, the same call a revision's `upgrade()` makes, and it
sent the `ALTER TABLE` that added the column. Here it ran on a database in memory, by hand. The rest
of the notebook runs it the way a project does: from a revision file, applied by the `alembic`
command, to a database that remembers which revisions it has had.


## Setup

Fourteen imports, one of them installed first where it is missing, and the college built from its
classes.

- `alembic` is not imported here: the notebook runs it as a command. `version` and
  `PackageNotFoundError`, from `importlib.metadata`, tell whether it is installed, and the cell
  prints its version. Colab does not have it, so there the cell installs 1.20.0, the version this
  notebook runs, with `pip`.
- `subprocess` and `sys` run the `alembic` command with this notebook's Python, `os` passes it three
  settings, and `shlex` prints each command the way a shell would read it
- `re` rewrites one line of `alembic.ini`
- `sqlalchemy` is the library itself, and the cell prints its version
- `text` sends SQL written out, and `select`, `func`, `insert`, `create_engine` and `event` build and
  read the college, with the rest of what the classes need
- `StaticPool`, from `sqlalchemy.pool`, is the pool the helper uses for a database in memory
- `date` is what the `Date` columns take and return
- `logging` carries the SQL an engine logs to `PrintStatements`
- `Path` names the files, and `shutil` removes the scratch folder at the start and at the end

Setup builds the college with `create_all`, as every notebook from **Many to Many** on does, so the
database has its tables before Alembic is ever run, which is how most projects meet Alembic.

Colab has SQLAlchemy installed, and this notebook runs version 2.0.54. Any 2.0 release runs it,
though an error may be worded a little differently. To match it exactly, run
`%pip install sqlalchemy==2.0.54` in a cell of its own, restart the session, and run this cell again.


In [1]:
import logging
import os
import re
import shlex
import shutil
import subprocess
import sys
from datetime import date
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

try:
    version("alembic")
except PackageNotFoundError:                                        # Colab has no Alembic: install the version this notebook runs
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore", "alembic==1.20.0"],
                   check=True)

import sqlalchemy
from sqlalchemy import (CheckConstraint, ForeignKey, MetaData, String, UniqueConstraint, create_engine, event, func, insert,
                        select, text)
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship
from sqlalchemy.pool import StaticPool

SCRATCH = Path("scratch")
shutil.rmtree(SCRATCH, ignore_errors=True)
SCRATCH.mkdir()
DATABASE = SCRATCH / "college.db"

NAMES = [
    "Ana Reyes", "Ben Okafor", "Chloe Martin", "Daniel Kim", "Elena Petrova", "Felix Wagner",
    "Grace Lin", "Hassan Ali", "Isabel Costa", "Jonas Berg", "Keiko Tanaka", "Liam Murphy",
    "Maya Patel", "Noah Andersen", "Olivia Brandt", "Pavel Novak", "Quinn Harper", "Rosa Delgado",
    "Sam Ito", "Tara Nilsen", "Umar Farouk", "Vera Kowalski", "Wes Carter", "Yara Haddad",
    "Aoife O'Brien",
]
PROGRAMS = ["Biology", "Computer Science", "Mathematics", "Psychology", "History"]
TERMS = [("Fall 2024", "2024-08-26"), ("Spring 2025", "2025-01-13"), ("Fall 2025", "2025-08-25"),
         ("Spring 2026", "2026-01-12")]
STUDENTS = [(name, f"{name[0]}{name.split()[-1]}@college.edu".lower().replace("'", ""),
             PROGRAMS[i % len(PROGRAMS)], TERMS[i % 3][1]) for i, name in enumerate(NAMES)]
COURSES = [
    ("BIO-101", "Introduction to Biology", "Biology", 4),
    ("CHE-110", "General Chemistry", "Chemistry", 4),
    ("MAT-120", "Calculus I", "Mathematics", 4),
    ("MAT-121", "Calculus II", "Mathematics", 4),
    ("CSC-101", "Programming I", "Computer Science", 3),
    ("CSC-201", "Data Structures", "Computer Science", 3),
    ("ENG-105", "Composition", "English", 3),
    ("HIS-110", "World History", "History", 3),
    ("PSY-101", "Introduction to Psychology", "Psychology", 3),
    ("STA-200", "Statistics", "Mathematics", 3),
]
GRADES = ["A", "A-", "B+", "B", "B-", "C+", "C", "C-", "D", "F"]

# One section of every course in every term, so the section of course c in term t has id (t - 1) * 10 + c.
SECTIONS = [(course, term, 30) for term in range(1, len(TERMS) + 1) for course in range(1, len(COURSES) + 1)]

# Three courses a term for every student, from the term they started. Spring 2026 is under way.
ENROLLMENTS = []
for s in range(len(NAMES)):
    for term in range(s % 3 + 1, len(TERMS) + 1):
        for k in range(3):
            section = (term - 1) * len(COURSES) + (s + term + 3 * k) % len(COURSES) + 1
            if term < len(TERMS):
                ENROLLMENTS.append((s + 1, section, "completed", GRADES[(s * 7 + term * 5 + k * 3) % len(GRADES)]))
            else:
                ENROLLMENTS.append((s + 1, section, "enrolled", None))

class PrintStatements(logging.Handler):
    """Print what an engine logs, leaving out the time: every statement, and the values sent with it."""

    def emit(self, record):
        if record.msg == "[%s] %r":                  # after a statement: how long it took, then its values
            values = repr(record.args[1])
            if values != "()":
                print("    values:", values)
        else:
            for line in record.getMessage().splitlines():
                print("   ", line.rstrip())


sql_log = logging.getLogger("sqlalchemy.engine.Engine")
sql_log.handlers = [PrintStatements()]              # this handler alone, however often the cell runs
sql_log.propagate = False                           # and no handler above it prints the same lines again


def college_engine(path=None, echo=False):
    """An engine for the college's database, in a file or in memory, with foreign keys enforced."""
    if path is None:                                # in memory: one connection, and one database, for every thread
        engine = create_engine("sqlite://", poolclass=StaticPool, echo=echo,
                               connect_args={"check_same_thread": False, "autocommit": False})
    else:
        engine = create_engine(f"sqlite:///{path}", echo=echo, connect_args={"autocommit": False})

    @event.listens_for(engine, "connect")
    def enforce_foreign_keys(dbapi_connection, connection_record):
        dbapi_connection.autocommit = True          # the PRAGMA does nothing inside a transaction,
        dbapi_connection.execute("PRAGMA foreign_keys = ON")
        dbapi_connection.autocommit = False         # and with autocommit=False sqlite3 keeps one open

    return engine

NAMING = {
    "pk": "pk_%(table_name)s",
    "uq": "uq_%(table_name)s_%(column_0_N_name)s",
    "ck": "ck_%(table_name)s_%(constraint_name)s",
    "fk": "fk_%(table_name)s_%(column_0_name)s_%(referred_table_name)s",
    "ix": "ix_%(column_0_label)s",
}


GRADE_POINTS = {"A": 4.0, "A-": 3.7, "B+": 3.3, "B": 3.0, "B-": 2.7, "C+": 2.3, "C": 2.0, "C-": 1.7, "D": 1.0, "F": 0.0}


class Base(DeclarativeBase):
    metadata = MetaData(naming_convention=NAMING)


class Student(Base):
    __tablename__ = "students"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(100))
    email: Mapped[str] = mapped_column(String(200), unique=True)
    program: Mapped[str] = mapped_column(String(50))
    started_on: Mapped[date]

    enrollments: Mapped[list["Enrollment"]] = relationship(back_populates="student", order_by="Enrollment.section_id")

    def __repr__(self):
        return f"Student({self.name!r}, {self.program!r})"


class Course(Base):
    __tablename__ = "courses"
    __table_args__ = (CheckConstraint("credits BETWEEN 1 AND 6", name="credits_range"),)

    id: Mapped[int] = mapped_column(primary_key=True)
    code: Mapped[str] = mapped_column(String(10), unique=True)
    title: Mapped[str] = mapped_column(String(100))
    department: Mapped[str] = mapped_column(String(50))
    credits: Mapped[int]

    sections: Mapped[list["Section"]] = relationship(back_populates="course", order_by="Section.term_id")

    def __repr__(self):
        return f"Course({self.code!r}, {self.credits})"


class Term(Base):
    __tablename__ = "terms"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(20), unique=True)
    starts_on: Mapped[date]

    sections: Mapped[list["Section"]] = relationship(back_populates="term", order_by="Section.course_id")

    def __repr__(self):
        return f"Term({self.name!r})"


class Section(Base):
    __tablename__ = "sections"
    __table_args__ = (UniqueConstraint("course_id", "term_id"), CheckConstraint("capacity > 0", name="capacity_positive"))

    id: Mapped[int] = mapped_column(primary_key=True)
    course_id: Mapped[int] = mapped_column(ForeignKey("courses.id"))
    term_id: Mapped[int] = mapped_column(ForeignKey("terms.id"))
    capacity: Mapped[int]

    course: Mapped["Course"] = relationship(back_populates="sections")
    term: Mapped["Term"] = relationship(back_populates="sections")
    enrollments: Mapped[list["Enrollment"]] = relationship(back_populates="section", order_by="Enrollment.student_id")

    def __repr__(self):
        return f"Section({self.id})"


class Enrollment(Base):
    __tablename__ = "enrollments"
    __table_args__ = (CheckConstraint("status IN ('enrolled', 'completed', 'withdrawn')", name="status_known"),)

    student_id: Mapped[int] = mapped_column(ForeignKey("students.id"), primary_key=True)
    section_id: Mapped[int] = mapped_column(ForeignKey("sections.id"), primary_key=True)
    status: Mapped[str] = mapped_column(String(20), server_default="enrolled")
    grade: Mapped[str | None] = mapped_column(String(2))

    student: Mapped["Student"] = relationship(back_populates="enrollments")
    section: Mapped["Section"] = relationship(back_populates="enrollments")

    @property
    def grade_points(self):
        """The points the grade is worth, or None before there is a grade."""
        return None if self.grade is None else GRADE_POINTS[self.grade]

    def __repr__(self):
        return f"Enrollment(student {self.student_id}, section {self.section_id}, {self.grade!r})"


def build_college(engine):
    """Create the college's tables from the classes, load the lists above into them, and count their rows."""
    Base.metadata.create_all(engine)
    rows = {
        Course: [{"code": code, "title": title, "department": department, "credits": credits}
                 for code, title, department, credits in COURSES],
        Student: [{"name": name, "email": email, "program": program, "started_on": date.fromisoformat(started)}
                  for name, email, program, started in STUDENTS],
        Term: [{"name": name, "starts_on": date.fromisoformat(starts)} for name, starts in TERMS],
        Section: [{"course_id": course, "term_id": term, "capacity": capacity} for course, term, capacity in SECTIONS],
        Enrollment: [{"student_id": student, "section_id": section, "status": status, "grade": grade}
                     for student, section, status, grade in ENROLLMENTS],
    }
    with engine.begin() as conn:
        for cls, values in rows.items():
            conn.execute(insert(cls), values)
        return {cls.__tablename__: conn.execute(select(func.count()).select_from(cls)).scalar_one() for cls in rows}


engine = college_engine(DATABASE)
print("sqlalchemy", sqlalchemy.__version__, "|", DATABASE, "|", build_college(engine))
print("alembic", version("alembic"))


sqlalchemy 2.0.54 | scratch/college.db | {'courses': 10, 'students': 25, 'terms': 4, 'sections': 40, 'enrollments': 228}
alembic 1.20.0


## Worked examples

### A migration environment for the college

Alembic runs as a command, in a process of its own, so it cannot see the classes this notebook
defined. A project keeps its models in a module, and so does this one. The next cell writes Setup's
classes to `scratch/college_models.py`:


In [2]:
%%writefile scratch/college_models.py
"""The college's tables, as classes: the models Alembic compares the database with."""
from datetime import date

from sqlalchemy import CheckConstraint, ForeignKey, MetaData, String, UniqueConstraint
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship

NAMING = {
    "pk": "pk_%(table_name)s",
    "uq": "uq_%(table_name)s_%(column_0_N_name)s",
    "ck": "ck_%(table_name)s_%(constraint_name)s",
    "fk": "fk_%(table_name)s_%(column_0_name)s_%(referred_table_name)s",
    "ix": "ix_%(column_0_label)s",
}


GRADE_POINTS = {"A": 4.0, "A-": 3.7, "B+": 3.3, "B": 3.0, "B-": 2.7, "C+": 2.3, "C": 2.0, "C-": 1.7, "D": 1.0, "F": 0.0}


class Base(DeclarativeBase):
    metadata = MetaData(naming_convention=NAMING)


class Student(Base):
    __tablename__ = "students"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(100))
    email: Mapped[str] = mapped_column(String(200), unique=True)
    program: Mapped[str] = mapped_column(String(50))
    started_on: Mapped[date]

    enrollments: Mapped[list["Enrollment"]] = relationship(back_populates="student", order_by="Enrollment.section_id")

    def __repr__(self):
        return f"Student({self.name!r}, {self.program!r})"


class Course(Base):
    __tablename__ = "courses"
    __table_args__ = (CheckConstraint("credits BETWEEN 1 AND 6", name="credits_range"),)

    id: Mapped[int] = mapped_column(primary_key=True)
    code: Mapped[str] = mapped_column(String(10), unique=True)
    title: Mapped[str] = mapped_column(String(100))
    department: Mapped[str] = mapped_column(String(50))
    credits: Mapped[int]

    sections: Mapped[list["Section"]] = relationship(back_populates="course", order_by="Section.term_id")

    def __repr__(self):
        return f"Course({self.code!r}, {self.credits})"


class Term(Base):
    __tablename__ = "terms"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(20), unique=True)
    starts_on: Mapped[date]

    sections: Mapped[list["Section"]] = relationship(back_populates="term", order_by="Section.course_id")

    def __repr__(self):
        return f"Term({self.name!r})"


class Section(Base):
    __tablename__ = "sections"
    __table_args__ = (UniqueConstraint("course_id", "term_id"), CheckConstraint("capacity > 0", name="capacity_positive"))

    id: Mapped[int] = mapped_column(primary_key=True)
    course_id: Mapped[int] = mapped_column(ForeignKey("courses.id"))
    term_id: Mapped[int] = mapped_column(ForeignKey("terms.id"))
    capacity: Mapped[int]

    course: Mapped["Course"] = relationship(back_populates="sections")
    term: Mapped["Term"] = relationship(back_populates="sections")
    enrollments: Mapped[list["Enrollment"]] = relationship(back_populates="section", order_by="Enrollment.student_id")

    def __repr__(self):
        return f"Section({self.id})"


class Enrollment(Base):
    __tablename__ = "enrollments"
    __table_args__ = (CheckConstraint("status IN ('enrolled', 'completed', 'withdrawn')", name="status_known"),)

    student_id: Mapped[int] = mapped_column(ForeignKey("students.id"), primary_key=True)
    section_id: Mapped[int] = mapped_column(ForeignKey("sections.id"), primary_key=True)
    status: Mapped[str] = mapped_column(String(20), server_default="enrolled")
    grade: Mapped[str | None] = mapped_column(String(2))

    student: Mapped["Student"] = relationship(back_populates="enrollments")
    section: Mapped["Section"] = relationship(back_populates="enrollments")

    @property
    def grade_points(self):
        """The points the grade is worth, or None before there is a grade."""
        return None if self.grade is None else GRADE_POINTS[self.grade]

    def __repr__(self):
        return f"Enrollment(student {self.student_id}, section {self.section_id}, {self.grade!r})"


Writing scratch/college_models.py


Two helpers do the rest of the notebook's work. `alembic` runs the command with this notebook's
Python, in the project folder, with three settings: `NO_COLOR`, since the kernel asks every program
it starts for color and a traceback would otherwise arrive full of terminal codes;
`PYTHONDONTWRITEBYTECODE`, so that a revision edited within a second of its last run is never read
from an old compiled copy; and `PYTHONUNBUFFERED`, so that what the command prints arrives in the
order it printed it. Before every command it closes the notebook's own connections, since SQLite can
let a connection that was open before a migration go on describing a table as it was. It takes the
project folder's path out of what it prints, keeps a traceback's last line only, and leaves out
three lines that every command prints the same. `edit` changes one
piece of a file, and refuses if the piece is not there exactly once:


In [3]:
PROJECT = Path("scratch")
MODELS = PROJECT / "college_models.py"
VERSIONS = PROJECT / "migrations" / "versions"


def alembic(*arguments):
    """Run an alembic command in the project folder, and print what it printed, less the lines every command repeats."""
    engine.dispose()                                                # the notebook's own connections close first
    done = subprocess.run(
        [sys.executable, "-m", "alembic", *arguments], cwd=PROJECT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, env={**os.environ, "NO_COLOR": "1", "PYTHONDONTWRITEBYTECODE": "1", "PYTHONUNBUFFERED": "1"},
    )
    lines = done.stdout.replace(f"{PROJECT.resolve()}{os.sep}", "").splitlines()
    if "Traceback (most recent call last):" in lines:              # the error's last line, not Python's own files
        lines = lines[:lines.index("Traceback (most recent call last):")] + ["Traceback (most recent call last): ...", lines[-1]]
    print("$", shlex.join(["alembic", *arguments]))
    for line in lines:
        if not any(noise in line for noise in ("Context impl", "Will assume", "setting up autogenerate plugin")):
            print("   ", line)


def edit(path, old, new):
    """Replace the one place in a file where old appears with new."""
    source = Path(path).read_text()
    assert source.count(old) == 1, f"{old!r} appears {source.count(old)} times in {path}"
    Path(path).write_text(source.replace(old, new))


alembic("init", "migrations")

ini = PROJECT / "alembic.ini"
ini.write_text(re.sub(r"^sqlalchemy\.url = .*$", "sqlalchemy.url = sqlite:///college.db", ini.read_text(), flags=re.M))
edit(PROJECT / "migrations" / "env.py", "target_metadata = None", "from college_models import Base\n\ntarget_metadata = Base.metadata")


$ alembic init migrations
    Creating directory migrations ...  done
    Creating directory migrations/versions ...  done
    Generating migrations/script.py.mako ...  done
    Generating migrations/env.py ...  done
    Generating migrations/README ...  done
    Generating alembic.ini ...  done
    Please edit configuration/connection/logging settings in alembic.ini before proceeding.


`alembic init` wrote `alembic.ini` and the `migrations` folder: `env.py`, which every command runs,
`script.py.mako`, the template for a new revision, a `README`, and `versions`, empty so far. Two
edits point the environment at the college. `sqlalchemy.url` in `alembic.ini` names the database,
relative to the project folder, and `target_metadata` in `env.py` is the `MetaData` that
autogenerate compares the database with. `env.py` can import `college_models` because `alembic.ini`
puts the folder the command runs in, the project folder, on Python's path.

### A first revision: the database as it is

The college's database already has every table the models describe, since Setup built it from the
same classes. The first revision is a baseline, and `print_revision` shows a revision's two
functions:


In [4]:
def revision_file(rev_id):
    """The file of the revision with this id."""
    return next(VERSIONS.glob(f"{rev_id}_*.py"))


def print_revision(rev_id):
    """Print a revision's upgrade and downgrade, leaving out its header, which holds the time it was written."""
    source = revision_file(rev_id).read_text()
    print(source[source.index("def upgrade"):].rstrip())


alembic("revision", "--autogenerate", "-m", "baseline", "--rev-id", "0001")
print_revision("0001")
alembic("stamp", "head")
alembic("current")


$ alembic revision --autogenerate -m baseline --rev-id 0001
    Generating migrations/versions/0001_baseline.py ...  done
def upgrade() -> None:
    """Upgrade schema."""
    # ### commands auto generated by Alembic - please adjust! ###
    pass
    # ### end Alembic commands ###


def downgrade() -> None:
    """Downgrade schema."""
    # ### commands auto generated by Alembic - please adjust! ###
    pass
    # ### end Alembic commands ###
$ alembic stamp head
    INFO  [alembic.runtime.migration] Running stamp_revision  -> 0001
$ alembic current
    0001 (head)


Autogenerate compared the models with the database and found nothing to change, so `upgrade()` and
`downgrade()` are both `pass`: the baseline is the revision the database already matches. `stamp`
wrote 0001 into `alembic_version`, a table of Alembic's own, without running anything, and `current`
read it back. `--rev-id` gives the revision an id of the notebook's choosing; without it, Alembic
makes up a random one, which would differ on every run of this notebook.

### A new column, from the model to the database

Students get a phone number. The model changes first, then autogenerate writes the revision, and the
revision is read before it runs:


In [5]:
edit(MODELS, "    started_on: Mapped[date]\n",
     "    started_on: Mapped[date]\n    phone: Mapped[str | None] = mapped_column(String(20))\n")
alembic("revision", "--autogenerate", "-m", "add phone", "--rev-id", "0002")
print_revision("0002")
alembic("upgrade", "head")

with engine.begin() as conn:
    conn.execute(text("UPDATE students SET phone = '555-0142' WHERE name = 'Ana Reyes'"))
with engine.connect() as conn:
    print(conn.execute(text("SELECT name, phone FROM students WHERE id <= 2")).all())


$ alembic revision --autogenerate -m 'add phone' --rev-id 0002
    INFO  [alembic.autogenerate.compare.tables] Detected added column 'students.phone'
    Generating migrations/versions/0002_add_phone.py ...  done
def upgrade() -> None:
    """Upgrade schema."""
    # ### commands auto generated by Alembic - please adjust! ###
    op.add_column('students', sa.Column('phone', sa.String(length=20), nullable=True))
    # ### end Alembic commands ###


def downgrade() -> None:
    """Downgrade schema."""
    # ### commands auto generated by Alembic - please adjust! ###
    op.drop_column('students', 'phone')
    # ### end Alembic commands ###
$ alembic upgrade head
    INFO  [alembic.runtime.migration] Running upgrade 0001 -> 0002, add phone
[('Ana Reyes', '555-0142'), ('Ben Okafor', None)]


Autogenerate reported the column it found, and the revision adds it in `upgrade()` and drops it in
`downgrade()`. `upgrade head` ran every revision after the one `alembic_version` held, which was
0002 alone. The new column takes values at once: Ana Reyes has a phone number, and Ben Okafor has
`None`. The number is from 555-0100 to 555-0199, the range kept for made-up numbers.

### Down and up again

`downgrade -1` undoes the latest revision, and `history` lists them all:


In [6]:
alembic("downgrade", "-1")
alembic("current")
print([column["name"] for column in sqlalchemy.inspect(engine).get_columns("students")])

alembic("upgrade", "head")
alembic("history")
with engine.connect() as conn:
    print(conn.execute(text("SELECT name, phone FROM students WHERE id = 1")).all())


$ alembic downgrade -1
    INFO  [alembic.runtime.migration] Running downgrade 0002 -> 0001, add phone
$ alembic current
    0001
['id', 'name', 'email', 'program', 'started_on']
$ alembic upgrade head
    INFO  [alembic.runtime.migration] Running upgrade 0001 -> 0002, add phone
$ alembic history
    0001 -> 0002 (head), add phone
    <base> -> 0001, baseline
[('Ana Reyes', None)]


The downgrade ran 0002's `downgrade()`, which dropped the column, and the upgrade added it back,
empty: Ana Reyes's number went with the column, and no upgrade brings data back. A downgrade is for
undoing a change on a database that has not come to depend on it, such as a copy on a laptop.
`history` lists the revisions newest first, each with the one it follows, and `<base>` is where the
chain starts.

### Batch mode: a foreign key SQLite cannot add in place

The college hires advisors, and every student can have one. That is a new table and a new column on
`students` with a foreign key, and SQLite cannot add a foreign key to a table that exists. So
`env.py` first gets `render_as_batch=True`, which makes autogenerate write every change to an
existing table in batch mode:


In [7]:
edit(PROJECT / "migrations" / "env.py", "connection=connection, target_metadata=target_metadata",
     "connection=connection, target_metadata=target_metadata, render_as_batch=True")
edit(MODELS, "class Student(Base):", """class Advisor(Base):
    __tablename__ = "advisors"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(100))


class Student(Base):""")
edit(MODELS, "    phone: Mapped[str | None] = mapped_column(String(20))\n",
     "    phone: Mapped[str | None] = mapped_column(String(20))\n"
     "    advisor_id: Mapped[int | None] = mapped_column(ForeignKey(\"advisors.id\"))\n")

alembic("revision", "--autogenerate", "-m", "add advisors", "--rev-id", "0003")
print_revision("0003")
alembic("upgrade", "head")

print([key["name"] for key in sqlalchemy.inspect(engine).get_foreign_keys("students")])
with engine.connect() as conn:
    print(conn.execute(select(func.count()).select_from(Student)).scalar(), "students, and broken foreign keys:",
          conn.execute(text("PRAGMA foreign_key_check")).all())


$ alembic revision --autogenerate -m 'add advisors' --rev-id 0003
    INFO  [alembic.autogenerate.compare.tables] Detected added table 'advisors'
    INFO  [alembic.autogenerate.compare.tables] Detected added column 'students.advisor_id'
    INFO  [alembic.autogenerate.compare.constraints] Detected added foreign key (advisor_id)(id) on table students
    Generating migrations/versions/0003_add_advisors.py ...  done
def upgrade() -> None:
    """Upgrade schema."""
    # ### commands auto generated by Alembic - please adjust! ###
    op.create_table('advisors',
    sa.Column('id', sa.Integer(), nullable=False),
    sa.Column('name', sa.String(length=100), nullable=False),
    sa.PrimaryKeyConstraint('id', name=op.f('pk_advisors'))
    )
    with op.batch_alter_table('students', schema=None) as batch_op:
        batch_op.add_column(sa.Column('advisor_id', sa.Integer(), nullable=True))
        batch_op.create_foreign_key(batch_op.f('fk_students_advisor_id_advisors'), 'advisors', ['advisor_

Autogenerate found a table, a column and a foreign key. The new table is an ordinary
`op.create_table`, and the changes to `students` are inside `op.batch_alter_table`: on SQLite, batch
mode builds a new table with the column and the foreign key, copies every row across, drops the old
table and gives the new one its name, the twelve steps of the **Changing a Schema** notebook done for
you. All twenty-five students came through, the foreign key has the name the models' naming
convention gave it, and `PRAGMA foreign_key_check` found no row pointing nowhere. The name matters as
well: the revision's `downgrade()` drops the foreign key by that name.

### The SQL without the database: --sql

`--sql` runs nothing. It writes the SQL a range of revisions would send, for someone to read, or to
run on a database this computer cannot reach:


In [8]:
alembic("upgrade", "0001:0002", "--sql")


$ alembic upgrade 0001:0002 --sql
    INFO  [alembic.runtime.migration] Generating static SQL
    INFO  [alembic.runtime.migration] Running upgrade 0001 -> 0002, add phone
    -- Running upgrade 0001 -> 0002
    
    ALTER TABLE students ADD COLUMN phone VARCHAR(20);
    
    UPDATE alembic_version SET version_num='0002' WHERE alembic_version.version_num = '0001';
    


The range `0001:0002` names both ends, since in this mode there is no database to ask where it is.
The SQL is the revision's `ALTER TABLE`, then the `UPDATE` that moves `alembic_version` along, the
same bookkeeping an online upgrade does.

### Which command for which job

| The job | The command |
|---|---|
| start a migration environment | `alembic init migrations` |
| write a revision from what the models changed | `alembic revision --autogenerate -m "..."` |
| write a revision by hand | `alembic revision -m "..."` |
| run every revision the database has not had | `alembic upgrade head` |
| undo the latest revision | `alembic downgrade -1` |
| record a revision as run, without running it | `alembic stamp head` |
| see where the database is, and every revision there is | `alembic current`, `alembic history` |
| write the SQL of a range of revisions | `alembic upgrade 0001:0002 --sql` |
| join two heads | `alembic merge heads -m "..."` |

For a change to the models, `revision --autogenerate` is the default, and the file it writes is a
draft: read it before `upgrade`, and correct it where autogenerate guessed. `revision` without
`--autogenerate` is for what the models cannot describe, such as moving data from one column to
another.

### A schema change in one call, finished

The pieces of this notebook in one function. `migrate` writes a revision from the models, shows it,
and applies it, and when the models and the database already agree, it removes the empty revision
instead of adding a step that does nothing:


In [9]:
def migrate(message, rev_id):
    """Write a revision from what the models changed and apply it, or remove it when nothing changed."""
    alembic("revision", "--autogenerate", "-m", message, "--rev-id", rev_id)
    source = revision_file(rev_id).read_text()
    if "op." not in source[source.index("def upgrade"):source.index("def downgrade")]:
        revision_file(rev_id).unlink()
        print("nothing to migrate: the empty revision is removed")
        return
    print_revision(rev_id)
    alembic("upgrade", "head")
    alembic("current")


migrate("check", "0004")

edit(MODELS, "    program: Mapped[str] = mapped_column(String(50))",
     "    program: Mapped[str] = mapped_column(String(50), index=True)")
migrate("index students by program", "0004")


$ alembic revision --autogenerate -m check --rev-id 0004
    Generating migrations/versions/0004_check.py ...  done
nothing to migrate: the empty revision is removed
$ alembic revision --autogenerate -m 'index students by program' --rev-id 0004
    INFO  [alembic.autogenerate.compare.constraints] Detected added index 'ix_students_program' on '('program',)'
    Generating migrations/versions/0004_index_students_by_program.py ...  done
def upgrade() -> None:
    """Upgrade schema."""
    # ### commands auto generated by Alembic - please adjust! ###
    with op.batch_alter_table('students', schema=None) as batch_op:
        batch_op.create_index(batch_op.f('ix_students_program'), ['program'], unique=False)

    # ### end Alembic commands ###


def downgrade() -> None:
    """Downgrade schema."""
    # ### commands auto generated by Alembic - please adjust! ###
    with op.batch_alter_table('students', schema=None) as batch_op:
        batch_op.drop_index(batch_op.f('ix_students_program'))

The first call found the models and the database in agreement, and left no revision behind. The
second found the index that `index=True` asks for, wrote it inside `batch_alter_table`, since
`render_as_batch` is on, and applied it.

### Where each part came from

| In `migrate` | What it relies on | The section that showed it |
|---|---|---|
| `alembic("revision", "--autogenerate", ...)` | the models compared with the database | A new column, from the model to the database |
| `unlink()` for a revision with no `op.` in `upgrade()` | an empty revision, whose functions are `pass` | A first revision: the database as it is |
| `print_revision(rev_id)` before the upgrade | a revision read before it runs | A new column, from the model to the database |
| `batch_op` in the revision | `render_as_batch=True` in `env.py` | Batch mode: a foreign key SQLite cannot add in place |
| `alembic("upgrade", "head")`, then `current` | `alembic_version`, the revision the database is at | A first revision: the database as it is |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlalchemy-deep-dive/17-migrations-with-alembic-solutions.ipynb).

**1.** Give `Course` a `description` column, a string of up to 500 characters that may be empty,
and take it to the database with a revision you read before it runs.


In [10]:
# your code here


**2.** Undo that revision, delete its file and take the column out of the model again, so that the
project is as it was, and show that the column is gone from the database.


In [11]:
# your code here


**3.** Write a revision by hand that adds a term, Fall 2026, starting on 2026-08-24, with
`op.execute`, and a `downgrade()` that deletes it. Apply it, and count the terms.


In [12]:
# your code here


**4.** Print the SQL of the revision from task 3 with `--sql`.


In [13]:
# your code here


**5.** Give every section a room: a `rooms` table with an `id` and a `name`, and a `room_id` on
`sections` that may be empty and refers to it. Take it to the database, and print the foreign keys of
`sections`.


In [14]:
# your code here


**6.** Downgrade to the baseline, print `current`, and upgrade to the head again.


In [15]:
# your code here


## Common errors

### FAILED: Target database is not up to date.


In [16]:
alembic("downgrade", "-1")
alembic("revision", "--autogenerate", "-m", "next change", "--rev-id", "0005")


$ alembic downgrade -1
    INFO  [alembic.runtime.migration] Running downgrade 0004 -> 0003, index students by program
$ alembic revision --autogenerate -m 'next change' --rev-id 0005
    ERROR [alembic.util.messaging] Target database is not up to date.
    FAILED: Target database is not up to date.


The database was one revision behind, and autogenerate refused to compare: a revision written from
it would have repeated the change the missing revision makes. The command failed and printed why,
and a failed command raises nothing in the notebook, so the cell finished. Bring the database up to
date first:


In [17]:
alembic("upgrade", "head")
alembic("current")


$ alembic upgrade head
    INFO  [alembic.runtime.migration] Running upgrade 0003 -> 0004, index students by program
$ alembic current
    0004 (head)


### NotImplementedError: No support for ALTER of constraints in SQLite dialect. Please refer to the batch mode feature which allows for SQLite migrations using a copy-and-move strategy.


In [18]:
alembic("revision", "-m", "unique advisor names", "--rev-id", "0005")
edit(revision_file("0005"), '"""Upgrade schema."""\n    pass',
     '"""Upgrade schema."""\n    op.create_unique_constraint(op.f("uq_advisors_name"), "advisors", ["name"])')
alembic("upgrade", "head")
alembic("current")


$ alembic revision -m 'unique advisor names' --rev-id 0005
    Generating migrations/versions/0005_unique_advisor_names.py ...  done
$ alembic upgrade head
    INFO  [alembic.runtime.migration] Running upgrade 0004 -> 0005, unique advisor names
    Traceback (most recent call last): ...
    NotImplementedError: No support for ALTER of constraints in SQLite dialect. Please refer to the batch mode feature which allows for SQLite migrations using a copy-and-move strategy.
$ alembic current
    0004


A revision written by hand gets no batch mode from `render_as_batch`, which only changes what
autogenerate writes. `op.create_unique_constraint` asked SQLite for an `ALTER TABLE` it does not
have, and Alembic stopped before sending anything: `current` is still 0004. Write the change in batch
mode, write the `downgrade()` too, and say the same in the model, with `unique=True`, so that the
models describe the database again:


In [19]:
edit(revision_file("0005"), 'op.create_unique_constraint(op.f("uq_advisors_name"), "advisors", ["name"])',
     'with op.batch_alter_table("advisors") as batch_op:\n'
     '        batch_op.create_unique_constraint(batch_op.f("uq_advisors_name"), ["name"])')
edit(revision_file("0005"), '"""Downgrade schema."""\n    pass',
     '"""Downgrade schema."""\n    with op.batch_alter_table("advisors") as batch_op:\n'
     '        batch_op.drop_constraint(batch_op.f("uq_advisors_name"), type_="unique")')
edit(MODELS, '__tablename__ = "advisors"\n\n    id: Mapped[int] = mapped_column(primary_key=True)\n'
     '    name: Mapped[str] = mapped_column(String(100))',
     '__tablename__ = "advisors"\n\n    id: Mapped[int] = mapped_column(primary_key=True)\n'
     '    name: Mapped[str] = mapped_column(String(100), unique=True)')
print_revision("0005")
alembic("upgrade", "head")
print([constraint["name"] for constraint in sqlalchemy.inspect(engine).get_unique_constraints("advisors")])


def upgrade() -> None:
    """Upgrade schema."""
    with op.batch_alter_table("advisors") as batch_op:
        batch_op.create_unique_constraint(batch_op.f("uq_advisors_name"), ["name"])


def downgrade() -> None:
    """Downgrade schema."""
    with op.batch_alter_table("advisors") as batch_op:
        batch_op.drop_constraint(batch_op.f("uq_advisors_name"), type_="unique")
$ alembic upgrade head
    INFO  [alembic.runtime.migration] Running upgrade 0004 -> 0005, unique advisor names
['uq_advisors_name']


### No error, and a column's data about to go: a rename read as a drop and an add


In [20]:
with engine.begin() as conn:
    conn.execute(text("UPDATE students SET phone = '555-0142' WHERE name = 'Ana Reyes'"))

edit(MODELS, "    phone: Mapped[str | None]", "    mobile: Mapped[str | None]")
alembic("revision", "--autogenerate", "-m", "rename phone", "--rev-id", "0006")
print_revision("0006")


$ alembic revision --autogenerate -m 'rename phone' --rev-id 0006
    INFO  [alembic.autogenerate.compare.tables] Detected added column 'students.mobile'
    INFO  [alembic.autogenerate.compare.tables] Detected removed column 'students.phone'
    Generating migrations/versions/0006_rename_phone.py ...  done
def upgrade() -> None:
    """Upgrade schema."""
    # ### commands auto generated by Alembic - please adjust! ###
    with op.batch_alter_table('students', schema=None) as batch_op:
        batch_op.add_column(sa.Column('mobile', sa.String(length=20), nullable=True))
        batch_op.drop_column('phone')

    # ### end Alembic commands ###


def downgrade() -> None:
    """Downgrade schema."""
    # ### commands auto generated by Alembic - please adjust! ###
    with op.batch_alter_table('students', schema=None) as batch_op:
        batch_op.add_column(sa.Column('phone', sa.VARCHAR(length=20), nullable=True))
        batch_op.drop_column('mobile')

    # ### end Alembic commands ##

The column was renamed in the model, and autogenerate, which compares what is there and not how it
got there, saw a column `mobile` that the database lacks and a column `phone` that the models lack.
The revision would add `mobile` empty and drop `phone` with Ana Reyes's number in it. Nothing has run
yet, which is why every revision is read first. Correct it to a rename, in both directions:


In [21]:
edit(revision_file("0006"),
     "batch_op.add_column(sa.Column('mobile', sa.String(length=20), nullable=True))\n        batch_op.drop_column('phone')",
     "batch_op.alter_column('phone', new_column_name='mobile')")
edit(revision_file("0006"),
     "batch_op.add_column(sa.Column('phone', sa.VARCHAR(length=20), nullable=True))\n        batch_op.drop_column('mobile')",
     "batch_op.alter_column('mobile', new_column_name='phone')")
print_revision("0006")
alembic("upgrade", "head")
with engine.connect() as conn:
    print(conn.execute(text("SELECT name, mobile FROM students WHERE id = 1")).all())


def upgrade() -> None:
    """Upgrade schema."""
    # ### commands auto generated by Alembic - please adjust! ###
    with op.batch_alter_table('students', schema=None) as batch_op:
        batch_op.alter_column('phone', new_column_name='mobile')

    # ### end Alembic commands ###


def downgrade() -> None:
    """Downgrade schema."""
    # ### commands auto generated by Alembic - please adjust! ###
    with op.batch_alter_table('students', schema=None) as batch_op:
        batch_op.alter_column('mobile', new_column_name='phone')

    # ### end Alembic commands ###
$ alembic upgrade head
    INFO  [alembic.runtime.migration] Running upgrade 0005 -> 0006, rename phone
[('Ana Reyes', '555-0142')]


### FAILED: Multiple head revisions are present for given argument 'head'; please specify a specific target revision, '<branchname>@head' to narrow to a specific head, or 'heads' for all heads


In [22]:
alembic("revision", "-m", "add rooms", "--rev-id", "0007", "--head", "0005", "--splice")
alembic("heads")
alembic("upgrade", "head")


$ alembic revision -m 'add rooms' --rev-id 0007 --head 0005 --splice
    Generating migrations/versions/0007_add_rooms.py ...  done
$ alembic heads
    0006 (head)
    0007 (head)
$ alembic upgrade head
    ERROR [alembic.util.messaging] Multiple head revisions are present for given argument 'head'; please specify a specific target revision, '<branchname>@head' to narrow to a specific head, or 'heads' for all heads
    FAILED: Multiple head revisions are present for given argument 'head'; please specify a specific target revision, '<branchname>@head' to narrow to a specific head, or 'heads' for all heads


Two people each wrote a revision after 0005 on branches of their own: one renamed the phone column,
as 0006, and the other started on rooms, as 0007. `--head 0005 --splice` makes 0007 the way the
second person's would arrive, with 0005 as its parent. Once both files are in `versions`, the chain
has two heads, and `upgrade head` cannot tell which comes first. `merge` writes a revision whose
parents are both heads, and the chain has one again:


In [23]:
alembic("merge", "heads", "-m", "merge rename and rooms", "--rev-id", "0008")
alembic("upgrade", "head")
alembic("history")


$ alembic merge heads -m 'merge rename and rooms' --rev-id 0008
    Generating migrations/versions/0008_merge_rename_and_rooms.py ...  done
$ alembic upgrade head
    INFO  [alembic.runtime.migration] Running upgrade 0005 -> 0007, add rooms
    INFO  [alembic.runtime.migration] Running upgrade 0006, 0007 -> 0008, merge rename and rooms
$ alembic history
    0006, 0007 -> 0008 (head) (mergepoint), merge rename and rooms
    0005 -> 0006, rename phone
    0005 -> 0007, add rooms
    0004 -> 0005 (branchpoint), unique advisor names
    0003 -> 0004, index students by program
    0002 -> 0003, add advisors
    0001 -> 0002, add phone
    <base> -> 0001, baseline


Last, the engine lets go of the file, and this cell removes the scratch folder, with the database,
the models and the migrations in it:


In [24]:
engine.dispose()
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


## Recap

- `alembic init` makes the environment, and `alembic.ini` and `env.py` point it at the database and
  at the models' `MetaData`.
- `revision --autogenerate` writes a revision from what the models changed, and the file is a draft
  to read, and correct, before `upgrade head` runs it.
- `alembic_version` records the revision the database is at; `stamp` sets it without running
  anything, and `downgrade` runs revisions backwards, dropping what they added.
- On SQLite, `render_as_batch=True` and `op.batch_alter_table` make the changes `ALTER TABLE` cannot,
  by copying the table.
- Autogenerate refuses a database that is behind, reads a rename as a drop and an add, and two
  revisions from one parent need `merge heads`.


## What is next

The **Four Databases, One Codebase** notebook takes the college's models to PostgreSQL, MySQL, SQL
Server and Oracle: the same statements compiled four ways, a migration written as SQL for each, and
the statements that still belong to one database.


---

&#8592; **Previous:** [Async SQLAlchemy](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlalchemy-deep-dive/16-async-sqlalchemy.ipynb)  &nbsp;·&nbsp;  [SQLAlchemy, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlalchemy-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Four Databases, One Codebase](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlalchemy-deep-dive/18-four-databases-one-codebase.ipynb) &#8594;
